[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/ddpm.ipynb)

In [ ]:
import os
import json
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)
print(f'Using device: {device} ({gpu_name})')
RESULTS_DIR = 'ddpm'
os.makedirs(RESULTS_DIR, exist_ok=True)
DATASETS_TO_RUN = ['cifar10', 'eurosat']
BATCH_SIZE = 128
NUM_TIMESTEPS = 1000
NUM_EPOCHS = 20
LR = 2e-4


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


DATASET_CONFIGS = {
    "cifar10": {"in_channels": 3, "image_size": 32},
    "eurosat": {"in_channels": 3, "image_size": 64},
    "stl10": {"in_channels": 3, "image_size": 64},
    "celeba": {"in_channels": 3, "image_size": 64},
    "mnist": {"in_channels": 1, "image_size": 32},
}


def get_ddpm_dataloaders(dataset_name="cifar10", batch_size=128, data_dir="./data", num_workers=2):
    """Load [-1, 1] scaled train and test DataLoaders for CIFAR-10, EuroSAT, STL-10, CelebA, or MNIST."""
    name = dataset_name.lower()
    if name not in DATASET_CONFIGS:
        raise ValueError(f"Unsupported dataset '{dataset_name}'. Choose from {list(DATASET_CONFIGS.keys())}.")

    cfg = DATASET_CONFIGS[name]
    image_size = cfg["image_size"]

    if name == "celeba":
        transform = transforms.Compose([
            transforms.CenterCrop(178),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: (x * 2.0) - 1.0),
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: (x * 2.0) - 1.0),
        ])

    if name == "cifar10":
        train_dataset = datasets.CIFAR10(root=data_dir, train=True, transform=transform, download=True)
        test_dataset = datasets.CIFAR10(root=data_dir, train=False, transform=transform, download=True)
    elif name == "eurosat":
        full_dataset = datasets.EuroSAT(root=data_dir, transform=transform, download=True)
        train_size = int(0.8 * len(full_dataset))
        test_size = len(full_dataset) - train_size
        generator = torch.Generator().manual_seed(42)
        train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=generator)
    elif name == "stl10":
        train_dataset = datasets.STL10(root=data_dir, split="train+unlabeled", transform=transform, download=True)
        test_dataset = datasets.STL10(root=data_dir, split="test", transform=transform, download=True)
    elif name == "celeba":
        train_dataset = datasets.CelebA(root=data_dir, split="train", transform=transform, download=True)
        test_dataset = datasets.CelebA(root=data_dir, split="test", transform=transform, download=True)
    else:
        train_dataset = datasets.MNIST(root=data_dir, train=True, transform=transform, download=True)
        test_dataset = datasets.MNIST(root=data_dir, train=False, transform=transform, download=True)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=num_workers, pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
    )
    return train_loader, test_loader, cfg


In [ ]:
import torch


class LinearNoiseScheduler:
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.num_timesteps = num_timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end

        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_cum_prod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1.0 - self.alpha_cum_prod)

    def add_noise(self, original, noise, t):
        # Forward transition kernel: q(x_t | x_0) = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
        sqrt_alpha_cum_prod = self.sqrt_alpha_cum_prod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cum_prod = self.sqrt_one_minus_alpha_cum_prod[t].view(-1, 1, 1, 1)
        return sqrt_alpha_cum_prod * original + sqrt_one_minus_alpha_cum_prod * noise

    def sample_prev_timestep(self, xt, noise_pred, t):
        # Reverse transition kernel: p_theta(x_{t-1} | x_t)
        x0 = (xt - (self.sqrt_one_minus_alpha_cum_prod[t] * noise_pred)) / self.sqrt_alpha_cum_prod[t]
        x0 = torch.clamp(x0, -1.0, 1.0)

        mean = xt - ((self.betas[t] * noise_pred) / self.sqrt_one_minus_alpha_cum_prod[t])
        mean = mean / torch.sqrt(self.alphas[t])

        if t == 0:
            return mean, x0
        else:
            variance = (1.0 - self.alpha_cum_prod[t - 1]) / (1.0 - self.alpha_cum_prod[t])
            variance = variance * self.betas[t]
            sigma = variance ** 0.5
            z = torch.randn_like(xt)
            return mean + sigma * z, x0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def get_time_embedding(time_steps, temb_dim):
    half_dim = temb_dim // 2
    factor = 10000 ** (
        torch.arange(start=0, end=half_dim, dtype=torch.float32, device=time_steps.device) / half_dim
    )
    t_emb = time_steps[:, None].repeat(1, half_dim) / factor
    t_emb = torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim=-1)
    return t_emb


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, down_sample=True, num_heads=4, use_attn=True):
        super().__init__()
        self.down_sample = down_sample
        self.use_attn = use_attn
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels),
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        if self.use_attn:
            self.attention_norm = nn.GroupNorm(8, out_channels)
            self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.down_sample_conv = (
            nn.Conv2d(out_channels, out_channels, kernel_size=4, stride=2, padding=1)
            if self.down_sample
            else nn.Identity()
        )

    def forward(self, x, t_emb):
        out = self.resnet_conv_first(x)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(x)

        if self.use_attn:
            batch_size, channels, h, w = out.shape
            in_attn = out.reshape(batch_size, channels, h * w)
            in_attn = self.attention_norm(in_attn).transpose(1, 2)
            out_attn, _ = self.attention(in_attn, in_attn, in_attn)
            out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
            out = out + out_attn

        return self.down_sample_conv(out)


class MidBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, num_heads=4):
        super().__init__()
        self.resnet_conv_first = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, in_channels),
                nn.SiLU(),
                nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
        ])
        self.t_emb_layers = nn.ModuleList([
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels)),
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels)),
        ])
        self.resnet_conv_second = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
        ])
        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
        ])

    def forward(self, x, t_emb):
        out = self.resnet_conv_first[0](x)
        out = out + self.t_emb_layers[0](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[0](out)
        out = out + self.residual_input_conv[0](x)

        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn).transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn

        resnet_input = out
        out = self.resnet_conv_first[1](out)
        out = out + self.t_emb_layers[1](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[1](out)
        out = out + self.residual_input_conv[1](resnet_input)
        return out


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, up_sample=True, num_heads=4, use_attn=True):
        super().__init__()
        self.up_sample = up_sample
        self.use_attn = use_attn
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels),
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        if self.use_attn:
            self.attention_norm = nn.GroupNorm(8, out_channels)
            self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.up_sample_conv = (
            nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=4, stride=2, padding=1)
            if self.up_sample
            else nn.Identity()
        )

    def forward(self, x, out_down, t_emb):
        x = self.up_sample_conv(x)
        x = torch.cat([x, out_down], dim=1)

        out = self.resnet_conv_first(x)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(x)

        if self.use_attn:
            batch_size, channels, h, w = out.shape
            in_attn = out.reshape(batch_size, channels, h * w)
            in_attn = self.attention_norm(in_attn).transpose(1, 2)
            out_attn, _ = self.attention(in_attn, in_attn, in_attn)
            out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
            out = out + out_attn

        return out


class UNet(nn.Module):
    def __init__(self, im_channels=3):
        super().__init__()
        self.down_channels = [64, 128, 256, 256]
        self.mid_channels = [256, 256, 256]
        self.t_emb_dim = 128
        self.down_sample = [True, True, False]
        self.attns = [False, True, True]

        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
        )
        self.conv_in = nn.Conv2d(im_channels, self.down_channels[0], kernel_size=3, padding=1)

        self.downs = nn.ModuleList([
            DownBlock(
                self.down_channels[i],
                self.down_channels[i + 1],
                self.t_emb_dim,
                down_sample=self.down_sample[i],
                num_heads=4,
                use_attn=self.attns[i],
            )
            for i in range(len(self.down_channels) - 1)
        ])

        self.mids = nn.ModuleList([
            MidBlock(
                self.mid_channels[i],
                self.mid_channels[i + 1],
                self.t_emb_dim,
                num_heads=4,
            )
            for i in range(len(self.mid_channels) - 1)
        ])

        self.ups = nn.ModuleList([
            UpBlock(
                self.down_channels[i] * 2,
                self.down_channels[i - 1] if i != 0 else 32,
                self.t_emb_dim,
                up_sample=self.down_sample[i],
                num_heads=4,
                use_attn=self.attns[i],
            )
            for i in reversed(range(len(self.down_channels) - 1))
        ])

        self.norm_out = nn.GroupNorm(8, 32)
        self.conv_out = nn.Conv2d(32, im_channels, kernel_size=3, padding=1)

    def forward(self, x, t):
        out = self.conv_in(x)
        t_emb = get_time_embedding(torch.as_tensor(t).long(), self.t_emb_dim)
        t_emb = self.t_proj(t_emb)

        down_outs = []
        for down in self.downs:
            down_outs.append(out)
            out = down(out, t_emb)

        for mid in self.mids:
            out = mid(out, t_emb)

        for up in self.ups:
            down_out = down_outs.pop()
            out = up(out, down_out, t_emb)

        out = self.norm_out(out)
        out = F.silu(out)
        return self.conv_out(out)


In [ ]:
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim


def train_ddpm(
    model,
    scheduler,
    train_loader,
    num_epochs=20,
    lr=2e-4,
    device="cpu",
    checkpoint_path="ddpm.pth",
    results_path=None,
):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = []

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        losses = []
        for images, _ in train_loader:
            images = images.to(device)
            noise = torch.randn_like(images)
            t = torch.randint(0, scheduler.num_timesteps, (images.shape[0],), device=device)

            noisy_images = scheduler.add_noise(images, noise, t)
            noise_pred = model(noisy_images, t)

            loss = criterion(noise_pred, noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        mean_loss = sum(losses) / len(losses)
        epoch_time = time.time() - t0
        history.append({
            "epoch": epoch + 1,
            "noise_mse_loss": round(mean_loss, 6),
            "epoch_time_sec": round(epoch_time, 2),
        })
        print(f"[DDPM] Epoch [{epoch + 1}/{num_epochs}] - Noise MSE Loss: {mean_loss:.6f} ({epoch_time:.2f}s)")

    if checkpoint_path:
        torch.save(model.state_dict(), checkpoint_path)

    if results_path:
        with open(results_path, "w") as f:
            json.dump({"ddpm": history}, f, indent=2)

    return model, history


In [ ]:
import torch
import matplotlib.pyplot as plt


def _to_display_image(tensor):
    img = tensor.cpu()
    if img.shape[0] == 1:
        return img.squeeze(0), "gray"
    return img.permute(1, 2, 0).clamp(0.0, 1.0), None


def plot_forward_diffusion(
    scheduler,
    train_loader,
    timesteps_to_show=None,
    device="cpu",
    save_path="forward_diffusion.png",
):
    if timesteps_to_show is None:
        timesteps_to_show = [0, 100, 300, 600, 999]

    sample_imgs, _ = next(iter(train_loader))
    x0 = sample_imgs[0:1].to(device)

    fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(10, 2.5))
    for idx, step in enumerate(timesteps_to_show):
        t_tensor = torch.tensor([step], device=device)
        noise = torch.randn_like(x0)
        xt = scheduler.add_noise(x0, noise, t_tensor)
        img_display = ((xt[0] + 1.0) / 2.0).clamp(0.0, 1.0)
        disp, cmap = _to_display_image(img_display)
        axes[idx].imshow(disp, cmap=cmap)
        axes[idx].set_title(f"t = {step}")
        axes[idx].axis("off")

    plt.suptitle("Forward Diffusion Process q(x_t | x_0)")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


@torch.no_grad()
def sample_ddpm(model, scheduler, num_samples=16, im_size=32, im_channels=3, device="cpu"):
    model.eval()
    xt = torch.randn((num_samples, im_channels, im_size, im_size), device=device)

    for i in reversed(range(scheduler.num_timesteps)):
        t_batch = torch.full((num_samples,), i, device=device, dtype=torch.long)
        noise_pred = model(xt, t_batch)
        xt, _ = scheduler.sample_prev_timestep(xt, noise_pred, i)

    imgs = torch.clamp(xt, -1.0, 1.0).cpu()
    imgs = (imgs + 1.0) / 2.0
    return imgs


def plot_ddpm_samples(generated_imgs, save_path="ddpm_samples.png", title="DDPM: Generated Samples from Pure Gaussian Noise"):
    num_samples = generated_imgs.shape[0]
    cols = 8
    rows = (num_samples + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 1.5 * rows))
    for i in range(num_samples):
        r, c = divmod(i, cols)
        ax = axes[r, c] if rows > 1 else axes[c]
        disp, cmap = _to_display_image(generated_imgs[i])
        ax.imshow(disp, cmap=cmap)
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
import json
import math
import time
import numpy as np
from scipy import linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import inception_v3, Inception_V3_Weights



def _gaussian_window(window_size=11, sigma=1.5, channels=3, device="cpu"):
    coords = torch.arange(window_size, dtype=torch.float32, device=device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = g[:, None] * g[None, :]
    return window_2d.expand(channels, 1, window_size, window_size).contiguous()


def compute_ssim(img1, img2, window_size=11):
    """Compute mean SSIM between two [0, 1] image batches."""
    channels = img1.size(1)
    window = _gaussian_window(window_size, 1.5, channels, img1.device)
    pad = window_size // 2

    mu1 = F.conv2d(img1, window, padding=pad, groups=channels)
    mu2 = F.conv2d(img2, window, padding=pad, groups=channels)

    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    sigma1_sq = F.conv2d(img1 * img1, window, padding=pad, groups=channels) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=pad, groups=channels) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=pad, groups=channels) - mu1_mu2

    c1, c2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / (
        (mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2)
    )
    return ssim_map.mean().item()


@torch.no_grad()
def evaluate_ddpm_reconstruction(model, scheduler, test_loader, t_noise=250, device="cpu", max_samples=128):
    """Corrupt test images to timestep t_noise and denoise back to t=0 to measure MSE, PSNR, and SSIM."""
    model.eval()
    mse_list, psnr_list, ssim_list = [], [], []
    seen = 0

    for images, _ in test_loader:
        images = images.to(device)
        noise = torch.randn_like(images)
        t_tensor = torch.full((images.size(0),), t_noise, device=device, dtype=torch.long)
        xt = scheduler.add_noise(images, noise, t_tensor)

        for step in reversed(range(t_noise + 1)):
            t_batch = torch.full((images.size(0),), step, device=device, dtype=torch.long)
            noise_pred = model(xt, t_batch)
            xt, _ = scheduler.sample_prev_timestep(xt, noise_pred, step)

        orig_01 = torch.clamp((images + 1.0) / 2.0, 0.0, 1.0)
        recon_01 = torch.clamp((xt + 1.0) / 2.0, 0.0, 1.0)

        mse = F.mse_loss(recon_01, orig_01, reduction="mean").item()
        psnr = 10.0 * math.log10(1.0 / max(mse, 1e-10))
        ssim = compute_ssim(orig_01, recon_01)

        mse_list.append(mse)
        psnr_list.append(psnr)
        ssim_list.append(ssim)

        seen += images.size(0)
        if seen >= max_samples:
            break

    return {
        "t_noise_level": t_noise,
        "mse": round(float(np.mean(mse_list)), 6),
        "psnr_db": round(float(np.mean(psnr_list)), 3),
        "ssim": round(float(np.mean(ssim_list)), 4),
    }


def _load_inception_models(device):
    weights = Inception_V3_Weights.DEFAULT
    clf = inception_v3(weights=weights).to(device).eval()
    feat_extractor = inception_v3(weights=weights).to(device).eval()
    feat_extractor.fc = nn.Identity()
    return feat_extractor, clf


def _preprocess_for_inception(imgs):
    if imgs.size(1) == 1:
        imgs = imgs.repeat(1, 3, 1, 1)
    imgs = F.interpolate(imgs, size=(299, 299), mode="bilinear", align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=imgs.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=imgs.device).view(1, 3, 1, 1)
    return (imgs - mean) / std


@torch.no_grad()
def _extract_inception_stats(images_tensor, feat_extractor, clf, batch_size=64, device="cpu"):
    feats, probs = [], []
    for i in range(0, images_tensor.size(0), batch_size):
        batch = images_tensor[i : i + batch_size].to(device)
        inp = _preprocess_for_inception(batch)
        feats.append(feat_extractor(inp).cpu().numpy())
        logits = clf(inp)
        probs.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(feats, axis=0), np.concatenate(probs, axis=0)


def compute_fid(real_feats, gen_feats):
    mu1, sigma1 = np.mean(real_feats, axis=0), np.cov(real_feats, rowvar=False)
    mu2, sigma2 = np.mean(gen_feats, axis=0), np.cov(gen_feats, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = float(diff @ diff + np.trace(sigma1 + sigma2 - 2.0 * covmean))
    return round(fid, 3)


def compute_inception_score(probs, splits=5):
    scores = []
    n = probs.shape[0]
    split_size = max(1, n // splits)
    for k in range(splits):
        part = probs[k * split_size : (k + 1) * split_size]
        if len(part) == 0:
            continue
        py = np.mean(part, axis=0, keepdims=True)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        scores.append(np.exp(np.mean(np.sum(kl, axis=1))))
    return round(float(np.mean(scores)), 3), round(float(np.std(scores)), 3)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)
    return {
        "total_params": total,
        "trainable_params": trainable,
        "model_size_mb": round(size_mb, 3),
    }


@torch.no_grad()
def benchmark_ddpm(
    model,
    scheduler,
    test_loader,
    im_size=32,
    im_channels=3,
    device="cpu",
    history=None,
    num_gen_samples=128,
    batch_size=64,
    t_noise=250,
):
    """Run Reconstruction (partial denoising), Generative Quality (FID/IS), and Compute Efficiency benchmarks."""
    model.eval()
    recon_metrics = evaluate_ddpm_reconstruction(
        model, scheduler, test_loader, t_noise=t_noise, device=device, max_samples=batch_size
    )
    param_metrics = count_parameters(model)

    gen_batches = []
    if device.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.time()
    for i in range(0, num_gen_samples, batch_size):
        cur_bs = min(batch_size, num_gen_samples - i)
        imgs = sample_ddpm(model, scheduler, num_samples=cur_bs, im_size=im_size, im_channels=im_channels, device=device)
        gen_batches.append(imgs)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t_start
    gen_tensor = torch.cat(gen_batches, dim=0)

    real_batches = []
    collected = 0
    for imgs, _ in test_loader:
        real_batches.append(torch.clamp((imgs + 1.0) / 2.0, 0.0, 1.0))
        collected += imgs.size(0)
        if collected >= num_gen_samples:
            break
    real_tensor = torch.cat(real_batches, dim=0)[:num_gen_samples]

    feat_extractor, clf = _load_inception_models(device)
    real_feats, _ = _extract_inception_stats(real_tensor, feat_extractor, clf, batch_size=64, device=device)
    gen_feats, gen_probs = _extract_inception_stats(gen_tensor, feat_extractor, clf, batch_size=64, device=device)

    fid = compute_fid(real_feats, gen_feats)
    is_mean, is_std = compute_inception_score(gen_probs)

    avg_epoch_time = (
        round(float(np.mean([h["epoch_time_sec"] for h in history])), 2) if history else None
    )

    return {
        "reconstruction": recon_metrics,
        "generative_quality": {
            "fid": fid,
            "inception_score_mean": is_mean,
            "inception_score_std": is_std,
            "num_eval_samples": num_gen_samples,
        },
        "compute_efficiency": {
            **param_metrics,
            "avg_epoch_time_sec": avg_epoch_time,
            "sampling_latency_ms_per_image": round((elapsed / num_gen_samples) * 1000.0, 4),
            "sampling_throughput_imgs_per_sec": round(num_gen_samples / max(elapsed, 1e-6), 2),
        },
    }


def save_benchmark_results(results, save_path="benchmark_results.json"):
    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)
    return results


In [ ]:
scheduler = LinearNoiseScheduler(num_timesteps=NUM_TIMESTEPS, beta_start=1e-4, beta_end=0.02, device=device)
all_training_results = {"runtime_gpu": gpu_name}
all_benchmark_results = {"runtime_gpu": gpu_name}

for ds_name in DATASETS_TO_RUN:
    print(f"\n==================== DATASET: {ds_name.upper()} ====================")
    train_loader, test_loader, cfg = get_ddpm_dataloaders(dataset_name=ds_name, batch_size=BATCH_SIZE)
    in_ch, img_sz = cfg["in_channels"], cfg["image_size"]

    plot_forward_diffusion(
        scheduler, train_loader, timesteps_to_show=[0, 100, 300, 600, 999],
        device=device, save_path=os.path.join(RESULTS_DIR, f"forward_diffusion_{ds_name}.png"),
    )

    model = UNet(im_channels=in_ch).to(device)
    model, ddpm_hist = train_ddpm(
        model, scheduler, train_loader, num_epochs=NUM_EPOCHS, lr=LR, device=device,
        checkpoint_path=os.path.join(RESULTS_DIR, f"ddpm_{ds_name}.pth"),
    )

    generated_imgs = sample_ddpm(
        model, scheduler, num_samples=16, im_size=img_sz, im_channels=in_ch, device=device
    )
    plot_ddpm_samples(
        generated_imgs, save_path=os.path.join(RESULTS_DIR, f"ddpm_{ds_name}_samples.png"),
        title=f"{ds_name.upper()} DDPM: Generated Samples from Pure Gaussian Noise",
    )

    ddpm_bench = benchmark_ddpm(
        model, scheduler, test_loader, im_size=img_sz, im_channels=in_ch,
        device=device, history=ddpm_hist, num_gen_samples=128, batch_size=64, t_noise=250,
    )

    all_training_results[ds_name] = ddpm_hist
    all_benchmark_results[ds_name] = ddpm_bench
    print(f"\n[{ds_name.upper()}] DDPM Benchmark:", json.dumps(ddpm_bench, indent=2))

with open(os.path.join(RESULTS_DIR, "training_results.json"), "w") as f:
    json.dump(all_training_results, f, indent=2)
save_benchmark_results(all_benchmark_results, save_path=os.path.join(RESULTS_DIR, "benchmark_results.json"))
